## FlyRank Internship · Backend Track · W4 · A4
### Auth-Login & protect - Python / FastAPI + Supabase lane

### Setup - the test double

```bash
pip install fastapi "uvicorn[standard]" supabase python-dotenv httpx

In [1]:
import uuid
import time as _time
from typing import Optional

from fastapi import FastAPI, Depends, Response
from fastapi.responses import JSONResponse
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.testclient import TestClient
from pydantic import BaseModel


class FakeAuthApiError(Exception):
    # Mirrors supabase-py's real `AuthApiError` -- raised on any failed auth call.
    def __init__(self, message: str):
        self.message = message
        super().__init__(message)


class FakeUser:
    def __init__(self, id: str, email: str, created_at: str):
        self.id = id
        self.email = email
        self.created_at = created_at


class FakeSession:
    def __init__(self, access_token: str, refresh_token: str):
        self.access_token = access_token
        self.refresh_token = refresh_token


class FakeAuthResponse:
    def __init__(self, user=None, session=None):
        self.user = user
        self.session = session


class FakeSupabaseAuth:
    """Stand-in for `create_client(...).auth`. In-memory only, for proving route logic."""

    def __init__(self):
        self._accounts: dict = {}   # email -> {"password": ..., "user": FakeUser}
        self._sessions: dict = {}   # access_token -> email
        self._roles: dict = {}      # email -> role, used by the Stretch 403 demo

    def sign_up(self, credentials: dict):
        email, password = credentials.get("email"), credentials.get("password")
        if email in self._accounts:
            raise FakeAuthApiError("User already registered")
        user = FakeUser(id=str(uuid.uuid4()), email=email, created_at="2026-08-03T00:00:00Z")
        self._accounts[email] = {"password": password, "user": user}
        self._roles[email] = "user"
        return FakeAuthResponse(user=user)

    def sign_in_with_password(self, credentials: dict):
        email, password = credentials.get("email"), credentials.get("password")
        record = self._accounts.get(email)
        if record is None or record["password"] != password:
            raise FakeAuthApiError("Invalid login credentials")
        token = f"fake-access-{uuid.uuid4()}"
        self._sessions[token] = email
        session = FakeSession(access_token=token, refresh_token=f"fake-refresh-{uuid.uuid4()}")
        return FakeAuthResponse(user=record["user"], session=session)

    def get_user(self, token: Optional[str] = None):
        if not token or token not in self._sessions:
            raise FakeAuthApiError("Invalid or expired token")
        email = self._sessions[token]
        return FakeAuthResponse(user=self._accounts[email]["user"])

    def sign_out(self, token: Optional[str] = None):
        # Simplification for this fake only -- see the real logout caveat in Stage 4 below.
        if token and token in self._sessions:
            del self._sessions[token]
        return None


fake_supabase_auth = FakeSupabaseAuth()

def clear_route(app: FastAPI, path: str, method: str):
    """FastAPI/Starlette match routes in registration order, first match wins -- so
    redefining a route in a later cell needs the old one removed first, or it just sits
    unreachable behind the new one."""
    method = method.upper()
    app.router.routes = [
        r for r in app.router.routes
        if not (getattr(r, "path", None) == path and method in getattr(r, "methods", set()))
    ]

print("Test double ready. Real methods it mirrors: sign_up, sign_in_with_password, get_user, sign_out")


Test double ready. Real methods it mirrors: sign_up, sign_in_with_password, get_user, sign_out


c:\python313\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


### Stage 0 - Set up Supabase as your server

*"Before you can guard a castle, you build the guard tower."* This stage is entirely
outside what a notebook can do on your behalf — it needs your email and your browser.

1. Create a free account at [supabase.com](https://supabase.com/) and spin up a new
   project (e.g. `Auth-Practice`). Takes a minute or two to provision.
2. In the **Supabase Dashboard** → **Project Settings → API**, copy your **Project URL**
   and **anon key** (the public key — safe client-side). **Never** use the `service_role`
   key here; it bypasses all security.
3. Create a git-ignored `.env`:
   ```
   SUPABASE_URL=your_project_url
   SUPABASE_KEY=your_anon_key
   PORT=8000
   ```
4. **One-time setting that saves you an hour:** in **Authentication → Sign In / Providers
   → Email**, turn **"Confirm email" off** — so a fresh signup can log in immediately
   during development. (Leave it on in production; it's a real security feature.)

**Checkpoint (on your machine):** running your server logs something like "connected to
Supabase" with no errors, and your `.env` is already in `.gitignore`.


### Stage 1 - Open auth: Sign Up & Log In
*"The front gates open. Let users register their keys and come back with them."*

`POST /auth/signup` forwards `{email, password}` to Supabase, validates both are present
(`400` if not), returns `201` with the user object on success. `POST /auth/login` does the
same for sign-in, returns `401` with a JSON error on bad credentials, `200` with the
access + refresh token on success.


In [3]:
def error(status_code: int, message: str) -> JSONResponse:
    return JSONResponse(status_code=status_code, content={"error": message})

app = FastAPI(title="Auth API", version="1.0")

class SignupRequest(BaseModel):
    email: Optional[str] = None
    password: Optional[str] = None

class LoginRequest(BaseModel):
    email: Optional[str] = None
    password: Optional[str] = None

@app.post("/auth/signup", status_code=201)
def signup(payload: SignupRequest):
    if not payload.email or not payload.password:
        return error(400, "email and password are required")
    try:
        result = fake_supabase_auth.sign_up({"email": payload.email, "password": payload.password})
    except FakeAuthApiError as e:
        return error(400, str(e))
    return JSONResponse(status_code=201, content={
        "id": result.user.id, "email": result.user.email, "created_at": result.user.created_at
    })

@app.post("/auth/login")
def login(payload: LoginRequest):
    if not payload.email or not payload.password:
        return error(400, "email and password are required")
    try:
        result = fake_supabase_auth.sign_in_with_password(
            {"email": payload.email, "password": payload.password}
        )
    except FakeAuthApiError:
        return error(401, "Invalid login credentials")
    return {
        "access_token": result.session.access_token,
        "refresh_token": result.session.refresh_token,
    }

client = TestClient(app)

s1 = client.post("/auth/signup", json={"email": "test@example.com", "password": "password123"})
print("POST /auth/signup                 ->", s1.status_code, s1.json())
assert s1.status_code == 201

s2 = client.post("/auth/signup", json={"email": "test@example.com"})  # no password
print("POST /auth/signup (no password)    ->", s2.status_code, s2.json())
assert s2.status_code == 400

l1 = client.post("/auth/login", json={"email": "test@example.com", "password": "password123"})
print("POST /auth/login                   ->", l1.status_code, l1.json())
assert l1.status_code == 200 and "access_token" in l1.json()
ACCESS_TOKEN = l1.json()["access_token"]

l2 = client.post("/auth/login", json={"email": "test@example.com", "password": "WRONG"})
print("POST /auth/login (wrong password)  ->", l2.status_code, l2.json())
assert l2.status_code == 401

print("\n Task 1 checkpoint passed.")


POST /auth/signup                 -> 201 {'id': 'ba6acf64-3e44-42b6-8d37-9cab8839795a', 'email': 'test@example.com', 'created_at': '2026-08-03T00:00:00Z'}
POST /auth/signup (no password)    -> 400 {'error': 'email and password are required'}
POST /auth/login                   -> 200 {'access_token': 'fake-access-20cd99a0-0525-4827-b644-7a40f0d1ccd4', 'refresh_token': 'fake-refresh-dce77c14-d130-4436-b57a-d2b10cd78f31'}
POST /auth/login (wrong password)  -> 401 {'error': 'Invalid login credentials'}

 Task 1 checkpoint passed.


In [2]:
# commiting checkpoint to git